In [1]:
import pandas as pd
import numpy as np

In [2]:
from gensim.models import Word2Vec

model = Word2Vec.load("data/models/node2vec_01.model")

In [3]:
nodes = list(model.wv.index_to_key)
embeddings = np.array([model.wv[node] for node in nodes])

In [4]:
import hdbscan
from sklearn.preprocessing import normalize

embeddings_norm = normalize(embeddings)
clusterer = hdbscan.HDBSCAN(metric='euclidean')
labels = clusterer.fit_predict(embeddings_norm)

cluster_df = pd.DataFrame({
    "Alert Type": nodes,
    "Cluster": labels
})

cluster_map = cluster_df.set_index("Alert Type")["Cluster"].to_dict()
cluster_map

{'CONNECTIVITY_PORT_STATE_OUTBOUND_USAGE': 4,
 'CONNECTIVITY_PORT_STATE_INBOUND_USAGE': 4,
 'MYSQL_SGBD_INDEX_DETAILS': -1,
 'KAFKA_ACTIVE_CONTROLLERS': 3,
 'OPENSHIFT_ETCD_ELECTIONS_PER_DAY': 0,
 'HOST_UNREACHABLE': 5,
 'HARDWARE_INTEGRITY_VIRTUAL_MACHINES_VM_GUEST_STATE': 5,
 'HARDWARE_INTEGRITY_VIRTUAL_MACHINES_VM_STATE': 5,
 'KAFKA_BYTES_OUT_PER_TOPIC': 3,
 'STORAGE_VOLUME_STATUS_KBYTES_PER_SECOND_MAX': 2,
 'STORAGE_VOLUME_STATUS_IDLE_PERCENT_AVG': 2,
 'STORAGE_VOLUME_STATUS_IOSZ_CUR': 2,
 'STORAGE_VOLUME_STATUS_IDLE_PERCENT_CUR': 2,
 'STORAGE_LINK_STATISTICS_XCB_SENT_PER_SECOND_CUR': 2,
 'STORAGE_VOLUME_STATUS_IO_PER_SECOND_MAX': 2,
 'STORAGE_VOLUME_STATUS_SVT_CUR': 2,
 'STORAGE_VOLUME_STATUS_IOSZ_AVG': 2,
 'STORAGE_VOLUME_STATUS_IO_PER_SECOND_CUR': 2,
 'STORAGE_VOLUME_STATUS_KBYTES_PER_SECOND_CUR': 2,
 'STORAGE_VOLUME_STATUS_IOSZ_QLEN': 2,
 'STORAGE_VOLUME_STATUS_IO_PER_SECOND_AVG': 2,
 'STORAGE_VOLUME_STATUS_KBYTES_PER_SECOND_AVG': 2,
 'STORAGE_VOLUME_STATUS_SVT_AVG': 2,
 'OPENS

In [5]:
from src.graphing.cluster_and_temporal_based import ClusterAndTemporalBased
from src.preprocess.sequence_preprocessor import SequencePreprocessor

data_df = pd.read_csv('data/raw/active_alarms_prod.csv')
prepocessor = SequencePreprocessor()
data_df = prepocessor.select_features(data_df)

nodes_df = prepocessor.group_by(data_df)

threshold = pd.Timedelta(minutes=5)
graphs_list = []
temporal_threshold = ClusterAndTemporalBased()
for node_df in nodes_df:
    graph = temporal_threshold.to_graph(node_df, cluster_map, threshold=threshold)
    graphs_list.append(graph)

In [11]:
import networkx as nx

num_of_alerts_by_node = []
for node_df, graph in zip(nodes_df, graphs_list):
    node_name = node_df['Node Name'][0]
    num_of_alerts_by_node.append({
        'Node Name': node_name,
        'Number Of Alerts': len(graph),
        'Number of Sub-Graphs': nx.number_weakly_connected_components(graph),
        'Graph\'s Density': nx.density(graph)
    })
num_of_alerts_by_node = pd.DataFrame(num_of_alerts_by_node)
num_of_alerts_by_node = num_of_alerts_by_node.sort_values('Number Of Alerts', ascending=False)
num_of_alerts_by_node

,Node Name,Number Of Alerts,Number of Sub-Graphs,Graph's Density
56,SPCTP-IFOHPP-STR04,2004,1,0.414405
615,spctp-v40hpp-svt231-14,399,130,0.002626
286,UC13-PJ041-BARONEZA,264,1,0.217191
773,SPCTP-V40HPP-SVT201-01,237,33,0.010119
253,SPCTP-IFOHPP-SWT10,163,1,0.496327
279,SPCTP-IFOHPP-SWT09,162,1,0.496243
628,SPPDZ-ACSDEL-MDB02,114,1,0.494411
738,SPCTP-SKYNEC-SRC05,106,2,0.490566
769,SPCTP-CDOVMW-SDB14,67,1,0.500000
810,SPPDZ-ACSDEL-MDB03,64,1,0.499256


In [13]:
from notebooks.utils.view_network import plot_graph

node_index = 400
graph = graphs_list[node_index]
node_df = nodes_df[node_index]

plot_graph(graph, f"graph_alerts_clusterized_{node_df['Node Name'][0]}")